# Plot the output of cellpose_intensities

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

## 0. Plot parameters

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.3 

settheme <- theme_minimal() +
  theme(
    text = element_text(family = "sans", size = FONT.SIZE),
    panel.background = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(colour = "black"),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 0),
    legend.position = "right",
    title = element_text(colour = "black", size = FONT.SIZE),
    plot.title = element_text(size = FONT.SIZE, face = "plain")
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition = c("Developed" = "#285F62", 
               "Poor Quality" = "#CA4F33")

## 1. Extract summary files

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping3/output/plots"

EXP = "EXP3"

if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
root_dir <- "/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping3/output/cellpose_intensities"

# Find matching CSVs in all subfolders
csv_paths <- list.files(
  path = root_dir,
  pattern = "\\.csv$",
  recursive = TRUE,
  full.names = TRUE
)

if (length(csv_paths) == 0) stop("No matching files found.")

# Read and bind
combined_df <- purrr::map_dfr(csv_paths, ~ readr::read_csv(.x, show_col_types = FALSE))


#rename colums and add new column for sample number
combined_df <- combined_df %>%
  rename(image = sample) %>%
  # Remove everything from the start (^) up to and including " - "
  mutate(sample = str_remove(image, "^.* - "))
combined_df$sample <- as.character(combined_df$sample)


In [ ]:
head(combined_df)

In [ ]:
unique(combined_df$image)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv("/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping3/sample_sheet.csv", show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- combined_df %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

In [ ]:
head(merged_df)

In [ ]:
#merge with cyto intensity
cyto_df <- read_csv("/ceph.groups/mshahbazi.grp/rsakata/HE_Nondeveloping3/output/cytomask_intensities/cyto_intensity.csv", show_col_types = FALSE)


# join in sample in sample sheet is contained in combined df sample
merged_df <- regex_left_join(
  merged_df,
  cyto_df,
  by = c("file_name" = "file"),
  ignore_case = TRUE
) 

In [ ]:
tbl <- merged_df %>%
  group_by(sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop") %>%
  arrange(desc(n_images))  # optional

tbl

In [ ]:
order_cond <- c("Developed","Poor Quality")

merged_df <- merged_df %>%
  mutate(condition = factor(condition, levels = order_cond))

In [ ]:
order_cond <- c("MS1206","MS950","MS966")

merged_df <- merged_df %>%
  mutate(sample_name = factor(sample_name, levels = order_cond))

## 2. Preprocess

In [ ]:
colnames(merged_df)

In [ ]:
# #Compute the normalised intensities
# merged_df$GATA3_norm = merged_df$Mean_GATA3/merged_df$Mean_dapi
# merged_df$GATA4_norm = merged_df$Mean_GATA4/merged_df$Mean_dapi
# merged_df$NANOG_norm = merged_df$Mean_NANOG/merged_df$Mean_dapi

In [ ]:
#Compute the normalised intensities
merged_df$GATA3_norm = (merged_df$Mean_GATA3-merged_df$GATA3_mean)/merged_df$Mean_dapi
merged_df$GATA4_norm = (merged_df$Mean_GATA4-merged_df$GATA4_mean)/merged_df$Mean_dapi
merged_df$NANOG_norm = (merged_df$Mean_NANOG-merged_df$NANOG_mean)/merged_df$Mean_dapi

## Check if dapi normalisation makes sense

In [ ]:
title = "gata3_dapi"

w <- 2.5
h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_GATA3, color = condition)) +
  geom_point(alpha = 0.6, size = 0.3) +
  labs(x = " Mean_dapi", y = " Mean_GATA3", title = "") +
  settheme +
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

In [ ]:
title = "nanog_dapi"

w <- 2.5
h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)

ggscatter = ggplot(merged_df, aes(x = Mean_dapi, y = Mean_NANOG, color = condition)) +
  geom_point(alpha = 0.6, size = 0.3) +
  labs(x = " Mean_dapi", y = " Mean_NANOG", title = "") +
  settheme+
  scale_color_manual(values=col_condition)
ggsave(plot = ggscatter, filename = sprintf("%s/%s.pdf",out_dir, title), w = w, h = h)
ggscatter

## Plot 

### A) Intensity threshold_hist

In [ ]:
plot_hist <- function(
  data,
  x = GATA3_norm,
  facet = sample_name,
  out_dir,
  title = "GATA3_norm_hist",
  w = 5, h = 5,
  bins = 30,
  binwidth = NULL,
  fill = "#6CD1D4",
  outline = "gray10",
  linewidth = 0.2,
  free_y = TRUE,
  vline_at = NULL,              # <— vertical line at this x
  vline_color = "gray30",
  vline_size = 0.5,
  vline_lty = "dashed"
) {
  x     <- rlang::enquo(x)
  facet <- rlang::enquo(facet)

  p <- ggplot(data, aes(x = !!x)) +
    (if (!is.null(binwidth))
       geom_histogram(binwidth = binwidth, boundary = 0, closed = "left",
                                na.rm = TRUE, fill = fill, color = outline, linewidth = linewidth)
     else
       geom_histogram(bins = bins, na.rm = TRUE,
                                fill = fill, color = outline, linewidth = linewidth)) +
    labs(x = "normalised intensity", y = "", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0),
      breaks = scales::pretty_breaks(n = 2)   # <— fewer ticks
    )  +
    facet_grid(rows = vars(!!facet),
                        scales = if (free_y) "free_y" else "fixed",
      switch = "y"            ) +
    theme(
      strip.text.y.left = element_text(angle = 0, hjust = 0),
      strip.background = element_blank(),
      strip.placement  = "outside",
      legend.position = "none"
    )

  if (!is.null(vline_at)) {
    p <- p + geom_vline(xintercept = vline_at,
                                 linetype = vline_lty,
                                 linewidth = vline_size,
                                 color = vline_color)
  }

  ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                  plot = p, width = w, height = h)
  p
}




### B) Intensity threshold jitter

In [ ]:
library(ggplot2)
library(rlang)

plot_jitter <- function(
  data,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir,
  title = "",
  w = 5, h = 3,
  palette = NULL,
  hline_at = NULL,                 # <— add a dotted horizontal line at this y
  hline_color = "gray30",
  hline_size = 0.5,
  hline_lty =  "dashed"
) {
  x <- rlang::enquo(x); y <- rlang::enquo(y); color <- rlang::enquo(color)

  p <- ggplot(data, aes(x = fct_rev(!!x), y = !!y, color = !!color)) +
    geom_jitter(width = 0.2, size = 0.5, alpha = 0.6, na.rm = TRUE) +
    labs(x = "", y = "normalised intensity", title = title) +
    settheme +
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) p <- p + scale_color_manual(values = palette)
  if (!is.null(hline_at)) p <- p + geom_hline(yintercept = hline_at, linetype = hline_lty,
                                              linewidth = hline_size, color = hline_color)

  ggsave(file.path(out_dir, sprintf("B_%s.pdf", title)), plot = p, width = w, height = h)
  p
}


In [ ]:
#NANOG
NANOG_thresh = 0.25

 w = 2.5
 h = 1.5
options(repr.plot.width=w, repr.plot.height=h)
ggjitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = NANOG_norm,
  color = condition,
  out_dir = out_dir,
  title = "NANOG_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = NANOG_thresh
)
ggjitter

w <- 3
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gghist <- plot_hist(
  data    = merged_df,
  x       = NANOG_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "NANOG_norm_hist",
  fill = "#FF2F92",
  w = w, h = h,
  vline_at = NANOG_thresh
)
gghist


In [ ]:
 w = 2.5
 h = 1.5
options(repr.plot.width=w, repr.plot.height=h)

GATA3_thresh = 0.3

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GATA3_norm,
  color = condition,
  out_dir = out_dir,
  title = "GATA3_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = GATA3_thresh
)
gg_jitter

w <- 2.5
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA3_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.1,
  title   = "GATA3_norm_hist",
  fill = "#6CD1D4",
  w = w, h = h,
  vline_at = GATA3_thresh)
gg_hist

In [ ]:
 w = 2.5
 h = 1.5
options(repr.plot.width=w, repr.plot.height=h)

GATA4_thresh = 0.3

gg_jitter <- plot_jitter(
  data = merged_df,
  x = sample_name,
  y = GATA4_norm,
  color = condition,
  out_dir = out_dir,
  title = "GATA4_norm_intensity",
  w = w, h = h,
  palette = col_condition,
  hline_at = GATA4_thresh
)
gg_jitter

w <- 2.5
h <- 3
options(repr.plot.width=w, repr.plot.height=h)

gg_hist <- plot_hist(
  data    = merged_df,
  x       = GATA4_norm,
  facet   = sample_name,
  out_dir = out_dir,
  binwidth = 0.05,
  title   = "GATA4_norm_hist",
  fill = "#6EA537",
  w = w, h = h,
  vline_at = GATA4_thresh
)
gg_hist

### C) %pos marker

In [ ]:
plot_pct_bar_points <- function(
  data,                               # e.g., summary_df
  pct = pct_GATA3,                    # <-- column with % values to plot
  sample = sample_name,               # sample/category column
  condition = condition,              # grouping/fill column
  out_dir,                    # folder to save (optional)
  title = NULL,                       # default built from pct col name if NULL
  palette = NULL,                     # named vector for fill
  w = 2.5, h = 1.5,
  y_max = 110,
  y_ticks = 5,
  bar_width = 0.6,
  point_size = 0.5,
  point_alpha = 0.7,
  jitter_width = 0.05
) {
  pct      <- enquo(pct)
  sample   <- enquo(sample)
  condition<- enquo(condition)

  # default title from pct column name if not supplied
  if (is.null(title)) {
    title <- paste0(as_label(pct), "+")
  }

  # per-sample means (by condition) for the chosen pct column
  means_df <- data %>%
    group_by(!!sample, !!condition) %>%
    summarise(mean_pct = mean(!!pct, na.rm = TRUE), .groups = "drop")

  # build plot (reverse sample order, flip coords)
  p <- ggplot(data, aes(x = fct_rev(!!sample), y = !!pct)) +
    geom_col(
      data = means_df,
      aes(y = mean_pct, fill = !!condition),
      width = bar_width
    ) +
    geom_point(
      size = point_size, alpha = point_alpha,
      position = position_jitter(width = jitter_width),
      na.rm = TRUE
    ) +
    labs(x = "", y = "%", title = title, fill = rlang::as_name(condition)) +
    settheme +
    scale_y_continuous(limits = c(0, y_max), expand = c(0, 0),
                       breaks = scales::pretty_breaks(y_ticks)) +
    coord_flip()+
    theme(legend.position = "none")

  if (!is.null(palette)) {
    p <- p + scale_fill_manual(values = palette)
  }

  safe_title <- gsub("[^[:alnum:]_\\-]+","_", title)
  ggsave(file.path(out_dir, sprintf("C_%s.pdf", safe_title)),
                  plot = p, width = w, height = h)
  options(repr.plot.width=w, repr.plot.height=h)
  p
}

In [ ]:
# thresholds (edit if you want different cutoffs)
thr <- list(
  GATA3_norm = GATA3_thresh,
  NANOG_norm = NANOG_thresh,
  GATA4_norm = GATA4_thresh
)

summary_df <- merged_df %>%
  group_by(image, sample_name, condition) %>%
  summarise(
    n = n(),
    pct_GATA3 = 100 * mean(GATA3_norm > thr$GATA3_norm, na.rm = TRUE),
    pct_NANOG = 100 * mean(NANOG_norm > thr$NANOG_norm, na.rm = TRUE),
    pct_GATA4 = 100 * mean(GATA4_norm > thr$GATA4_norm, na.rm = TRUE),
    pct_negative = 100 * mean(!(GATA4_norm > thr$GATA4_norm | NANOG_norm > thr$NANOG_norm | GATA3_norm > thr$GATA3_norm)),
    pct_double_GATA3_NANOG = 100 * mean(GATA3_norm > thr$GATA3_norm & NANOG_norm > thr$NANOG_norm),
    .groups = "drop"
  )

head(summary_df)


In [ ]:
# Plot % GATA3+
plot_pct_bar_points(summary_df, pct = pct_GATA3, palette = col_condition, out_dir = out_dir, 
                    title = "pctGATA3+")

# Plot % NANOG+
plot_pct_bar_points(summary_df, pct = pct_NANOG, palette = col_condition,out_dir = out_dir,
                    title = "pctNANOG+")
# Plot % GATA4+
plot_pct_bar_points(summary_df, pct = pct_GATA4, palette = col_condition,out_dir = out_dir,
                    title = "pctGATA4+")

# Plot % neg
plot_pct_bar_points(summary_df, pct = pct_negative, palette = col_condition,out_dir = out_dir, 
                    title = "pctnegative")

# Plot % double+
plot_pct_bar_points(summary_df, pct = pct_double_GATA3_NANOG, palette = col_condition,out_dir = out_dir, 
                    title = "pct_GATA3+_NANOG+")



### D) Intensity per subset

In [ ]:
# find average intensities after subtyping
merged_df <- merged_df %>%
  mutate(
    GATA3pos = GATA3_norm > GATA3_thresh,
    NANOGpos = NANOG_norm > NANOG_thresh,
    GATA4pos = GATA4_norm > GATA4_thresh
  )


In [ ]:
title   <- "GATA3_norm_intensity_in GATA3+"
w <- 2.5; h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA3pos == TRUE), aes(x = forcats::fct_rev(sample_name), y = GATA3_norm, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA3 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")

ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "NANOG_norm_intensity_in_NANOGpos"
w <- 2.5; h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(NANOGpos == TRUE), aes(x = forcats::fct_rev(sample_name), y = NANOG_norm, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "NANOG intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")


ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p


In [ ]:
title   <- "GATA4_norm_intensity_in_GATA4pos"
w <- 2.5; h <- 1.5
options(repr.plot.width=w, repr.plot.height=h)
palette <- col_condition

p <- ggplot(merged_df %>% filter(GATA4pos == TRUE), aes(x = forcats::fct_rev(sample_name), y = GATA4_norm, color = condition)) +
  # boxplots per condition (behind points)
  geom_boxplot(
    aes(fill = condition),
    width = 0.5, alpha = 0.25, outlier.shape = NA,
    position = position_dodge2(width = 0.5, preserve = "single")
  ) +
  # points with jitter+dodge (no width/height here)
  geom_point(
    size = 0.5, alpha = 0.6,
    position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.5),
    na.rm = TRUE
  ) +
  labs(x = "", y = "GATA4 intensity (normalized)", title = title) +
  settheme +
  #scale_y_continuous(limits = c(0, NA), expand = c(0, 0)) +
  coord_flip() +
  scale_color_manual(values = palette) +
  scale_fill_manual(values = palette, guide = "none")  +
  theme(legend.position = "none")


ggsave(file.path(out_dir, sprintf("D_%s.pdf", title)), plot = p, width = w, height = h)
p

### E) Normalise intensity by average of control

In [ ]:
head(merged_df)

In [ ]:
# 1. Calculate the average values for the "Developed" condition
ctrl_vals <- merged_df %>%
  filter(condition == "Developed") %>%
  summarise(
    mean_NANOG = mean(NANOG_norm, na.rm = TRUE),
    mean_GATA3 = mean(GATA3_norm, na.rm = TRUE),
    mean_GATA4 = mean(GATA4_norm, na.rm = TRUE)
  )

# 2. Add these averages as columns and create the normalized columns
merged_df <- merged_df %>%
  mutate(
    ctrl_NANOG = ctrl_vals$mean_NANOG,
    ctrl_GATA3 = ctrl_vals$mean_GATA3,
    ctrl_GATA4 = ctrl_vals$mean_GATA4,
    
    # Create the ratio columns
    NANOG_norm_ctr = NANOG_norm / ctrl_NANOG,
    GATA3_norm_ctr = GATA3_norm / ctrl_GATA3,
    GATA4_norm_ctr = GATA4_norm / ctrl_GATA4
  )

# Check the results
head(merged_df)

## Save 

In [ ]:
merged_df$EXP = EXP
write_csv(merged_df, file.path(out_dir, "summarised_results.csv"))